# Using the TensorDataset class

In practice, loading your data into a PyTorch dataset will be one of the first steps you take in order to create and train a neural network with PyTorch.

The TensorDataset class is very helpful when your dataset can be loaded directly as a NumPy array. Recall that TensorDataset() can take one or more NumPy arrays as input.

In this exercise, you'll practice creating a PyTorch dataset using the TensorDataset class.

In [1]:
import pandas as pd
dataframe = pd.read_csv("dataset/water_potability.csv")
dataframe.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,0.587349,0.577747,0.386298,0.568199,0.647347,0.292985,0.654522,0.795029,0.630115,0
1,0.643654,0.441300,0.314381,0.439304,0.514545,0.356685,0.377248,0.202914,0.520358,0
2,0.388934,0.470876,0.506122,0.524364,0.561537,0.142913,0.249922,0.401487,0.219973,0
3,0.725820,0.715942,0.506141,0.521683,0.751819,0.148683,0.467200,0.658678,0.242428,0
4,0.610517,0.532588,0.237701,0.270288,0.495155,0.494792,0.409721,0.469762,0.585049,0


In [2]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

np_features = np.array(np.random.rand(12, 8))
np_target = np.array(np.random.rand(12, 1))

# Convert arrays to PyTorch tensors
torch_features = torch.from_numpy(np_features)
torch_target = torch.from_numpy(np_target)

# Create a TensorDataset from two tensors
dataset = TensorDataset(torch_features, torch_target)

# Return the last element of this dataset
print(dataset[-1])

(tensor([0.0556, 0.4789, 0.6199, 0.6956, 0.0547, 0.1504, 0.1083, 0.2216],
       dtype=torch.float64), tensor([0.1222], dtype=torch.float64))


# From data loading to running a forward pass

n this exercise, you'll create a PyTorch DataLoader from a pandas DataFrame and call a model on this dataset. Specifically, you'll run a forward pass on a neural network. You'll continue working with fully connected neural networks, as you have done so far.

You'll begin by subsetting a loaded DataFrame called dataframe, converting features and targets NumPy arrays, and converting to PyTorch tensors in order to create a PyTorch dataset

In [3]:
# Load the different columns into two PyTorch tensors
features = torch.tensor(dataframe[['ph', 'Sulfate', 'Conductivity', 'Organic_carbon']].to_numpy()).float()
target = torch.tensor(dataframe['Potability'].to_numpy()).float()

# Create a dataset from the two generated tensors
dataset = TensorDataset(features, target)

# Create a dataloader using the above dataset
dataloader = DataLoader(dataset, batch_size=3, shuffle=True)
# x, y = next(iter(dataloader))


In [4]:
from torch import nn
# Create a model using the nn.Sequential API
model = nn.Sequential(nn.Linear(4,1),
                      nn.Linear(1,1))
output = model(features)
print(output)

tensor([[-0.2340],
        [-0.2494],
        [-0.2914],
        ...,
        [-0.1701],
        [-0.2204],
        [-0.1401]], grad_fn=<AddmmBackward0>)


# Writing the evaluation loop

In this exercise, you will practice writing the evaluation loop. Recall that the evaluation loop is similar to the training loop, except that you will not perform the gradient calculation and the optimizer step.

In [5]:
# Load the different columns into two PyTorch tensors
validation_features = torch.tensor(dataframe[['ph', 'Sulfate', 'Conductivity', 'Organic_carbon']].iloc[-50:].to_numpy()).float()
validation_target = torch.tensor(dataframe[['Potability']].iloc[-50:].to_numpy()).float()

# Create a dataset from the two generated tensors
validation_dataset = TensorDataset(validation_features, validation_target)

# Create a dataloader using the above dataset
validationloader = DataLoader(validation_dataset, shuffle=True)


In [6]:
from torch.nn import CrossEntropyLoss, MSELoss, L1Loss
criterion = CrossEntropyLoss() 
# Set the model to evaluation mode
model.eval()
validation_loss = 0.0

with torch.no_grad():
  
  for data in validationloader:
    
      outputs = model(data[0])
      loss = criterion(outputs, data[1])
      
      # Sum the current loss to the validation_loss variable
      validation_loss += loss.item()
      
# Calculate the mean loss value
validation_loss_epoch = validation_loss/len(validationloader) # For 2 batches
print(validation_loss_epoch)

# Set the model back to training mode
model.train()

0.0


Sequential(
  (0): Linear(in_features=4, out_features=1, bias=True)
  (1): Linear(in_features=1, out_features=1, bias=True)
)

# Calculating accuracy using torchmetrics

In addition to the losses, you should also be keeping track of the accuracy during training. By doing so, you will be able to select the epoch when the model performed the best.

In this exercise, you will practice using the torchmetrics package to calculate the accuracy. You will be using a sample of the facemask dataset. This dataset contains three different classes. The plot_errors function will display samples where the model predictions do not match the ground truth. Performing such error analysis will help you understand your model failure modes.

In [7]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

def plot_errors(model, dataloader): 
    # find mismatches
    mismatches = []
    for data in dataloader:
        if len(mismatches) > 8:
            break
        features, labels = data
        outputs = model(features)
        gt = labels.argmax(-1)
        pred = outputs.argmax(-1)
        for f, g, p in zip(features, gt, pred):
            if g != p:
                mismatches.append((f, g, p))
    
    
    fig = plt.figure(figsize=(8, 8))
    grid = ImageGrid(fig, 111,  # similar to subplot(111)
                     nrows_ncols=(2, 4),  # creates 2x2 grid of axes
                     axes_pad=0.5,  # pad between axes in inch.
                     )
    mapping = {0: 'No mask', 1: 'Mask', 2: 'Incorrect'}
    for idx, ax in enumerate(grid):
        ax.imshow(mismatches[idx][0].permute(1, 2, 0))
        ax.set_title(f'GT: {mapping[mismatches[idx][1].item()]} \n PRED: {mapping[mismatches[idx][2].item()]}')
        ax.axis('off')
    plt.show()

In [8]:
import torchmetrics
# Create accuracy metric using torch metrics
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
for data in dataloader:
    features, labels = data
    outputs = model(features)
    
    # Calculate accuracy over the batch
    preds = torch.argmax(outputs, dim=1)
    acc = metric(preds, labels)
    
# Calculate accuracy over the whole epoch
acc = metric.compute()
print(acc.item())

# Reset the metric for the next epoch 
metric.reset()
# plot_errors(model, dataloader)

0.5967180728912354


# Experimenting with dropout

The dropout layer randomly zeroes out elements of the input tensor. Doing so helps fight overfitting. In this exercise, you'll create a small neural network with at least two linear layers, two dropout layers, and two activation functions.

In [9]:
# Create a small neural network
model = nn.Sequential(
    nn.Linear(3072, 16),
    nn.ReLU(),
    nn.Dropout(p=0.5)
)
# model(input_tensor)

In [10]:
# Create a small neural network
model = nn.Sequential(
    nn.Linear(3072, 16),
    nn.ReLU(),
    nn.Dropout(p=0.8)
)
# model(input_tensor)

# Understanding overfitting

Overfitting is very common in machine learning, where the trend is for bigger and bigger models. As a machine learning practitioner, you will face overfitting in your career. Which of the following statements about overfitting are true?

- Overfitting happens when the model is performing worse on the validation set than on the training set.
- Data augmentation can reduce overfitting by artificially increasing the size of the training set.
- A dropout layer with a probability strictly superior to zero will reduce overfitting.

# Implementing random search

Hyperparameter search is a computationally costly approach to experiment with different hyperparameter values. However, it can lead to performance improvements. In this exercise, you will implement a random search algorithm.

In [11]:
values = []
for idx in range(10):
    # Randomly sample a learning rate factor between 2 and 4
    factor = np.random.uniform(2, 4)
    lr = 10 ** -factor
    
    # Randomly select a momentum between 0.85 and 0.99
    momentum = np.random.uniform(0.85,0.99)
    
    values.append((lr, momentum))